# Import SQLite + Pandas

In [1]:
import sqlite3
import pandas as pd

# create database
conn = sqlite3.connect("sql_assignment.db")

cursor = conn.cursor()

print("Database connected")

Database connected


# 1. Create Tables

## Customers Table

In [2]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers(
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    email TEXT,
    city TEXT
)
""")

conn.commit()

## Products Table

In [3]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS products(
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT,
    price REAL
)
""")

conn.commit()

## Orders Table

In [4]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders(
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    
    FOREIGN KEY(customer_id)
    REFERENCES customers(customer_id)
)
""")

conn.commit()

## Order Items Table

In [5]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS order_items(
    item_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,

    FOREIGN KEY(order_id)
    REFERENCES orders(order_id),

    FOREIGN KEY(product_id)
    REFERENCES products(product_id)
)
""")

conn.commit()

# Create Indexes

In [6]:
cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_customer_city
ON customers(city)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_product_category
ON products(category)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_order_customer
ON orders(customer_id)
""")


conn.commit()

print("Indexes created")

Indexes created


# Insert Sample Data

### Customers

In [7]:
customers = [
(1,"Amit","amit@gmail.com","Pune"),
(2,"Sneha","sneha@gmail.com","Mumbai"),
(3,"Rahul","rahul@gmail.com","Delhi"),
(4,"Priya","priya@gmail.com","Pune")
]


cursor.executemany("""
INSERT OR IGNORE INTO customers
VALUES(?,?,?,?)
""",customers)

conn.commit()

### Products

In [8]:
products=[
(1,"Laptop","Electronics",60000),
(2,"Phone","Electronics",30000),
(3,"Chair","Furniture",5000),
(4,"Table","Furniture",10000)
]


cursor.executemany("""
INSERT OR IGNORE INTO products
VALUES(?,?,?,?)
""",products)


conn.commit()

### Orders

In [10]:
orders=[
(1,1,"2026-01-10"),
(2,2,"2026-01-15"),
(3,1,"2026-02-01")
]


cursor.executemany("""
INSERT OR IGNORE INTO orders
VALUES(?,?,?)
""",orders)

conn.commit()

### Order Items

In [11]:
items=[
(1,1,1,2),
(2,1,2,1),
(3,2,3,4),
(4,3,4,2)
]


cursor.executemany("""
INSERT OR IGNORE INTO order_items
VALUES(?,?,?,?)
""",items)

conn.commit()

# Validate Data

In [12]:
tables=[
"customers",
"products",
"orders",
"order_items"
]


for table in tables:
    result=pd.read_sql_query(
        f"SELECT COUNT(*) as count FROM {table}",
        conn
    )
    print(table)
    display(result)

customers


,count
0,4


products


,count
0,4


orders


,count
0,3


order_items


,count
0,4


# Section A – Basics

## Q1. Write a query to display all columns and rows from the customer's table.

In [13]:
pd.read_sql_query(
"""
SELECT *
FROM customers
""",
conn
)

,customer_id,customer_name,email,city
0,1,Amit,amit@gmail.com,Pune
1,2,Sneha,sneha@gmail.com,Mumbai
2,3,Rahul,rahul@gmail.com,Delhi
3,4,Priya,priya@gmail.com,Pune


## Q2. Retrieve only the first_name, last_name, and city of all customers.

In [17]:
pd.read_sql_query(
"""
SELECT 
customer_name,
city
FROM customers;
""",
conn
)

,customer_name,city
0,Amit,Pune
1,Sneha,Mumbai
2,Rahul,Delhi
3,Priya,Pune


## Q3. List all unique categories available in the products table.

In [18]:
pd.read_sql_query(
"""
SELECT DISTINCT category
FROM products;
""",
conn
)

,category
0,Electronics
1,Furniture


## Q4. Identify the Primary Key of each table in the schema. Explain why a Primary Key must be unique and NOT NULL.

In [19]:
tables = [
    "customers",
    "products",
    "orders",
    "order_items"
]

for table in tables:
    print("\nTable:", table)
    
    result = pd.read_sql_query(
    f"""
    PRAGMA table_info({table});
    """,
    conn
    )
    
    display(result)


Table: customers


,cid,name,type,notnull,dflt_value,pk
0,0,customer_id,INTEGER,0,None,1
1,1,customer_name,TEXT,1,None,0
2,2,email,TEXT,0,None,0
3,3,city,TEXT,0,None,0



Table: products


,cid,name,type,notnull,dflt_value,pk
0,0,product_id,INTEGER,0,None,1
1,1,product_name,TEXT,1,None,0
2,2,category,TEXT,0,None,0
3,3,price,REAL,0,None,0



Table: orders


,cid,name,type,notnull,dflt_value,pk
0,0,order_id,INTEGER,0,None,1
1,1,customer_id,INTEGER,0,None,0
2,2,order_date,TEXT,0,None,0



Table: order_items


,cid,name,type,notnull,dflt_value,pk
0,0,item_id,INTEGER,0,None,1
1,1,order_id,INTEGER,0,None,0
2,2,product_id,INTEGER,0,None,0
3,3,quantity,INTEGER,0,None,0


## Q5. What constraints are applied to the email column in the customers table? What happens if you insert a duplicate email?

In [20]:
pd.read_sql_query(
"""
PRAGMA table_info(customers);
""",
conn
)


,cid,name,type,notnull,dflt_value,pk
0,0,customer_id,INTEGER,0,None,1
1,1,customer_name,TEXT,1,None,0
2,2,email,TEXT,0,None,0
3,3,city,TEXT,0,None,0


# Q6. Try inserting a product with unit_price = -50. What happens and which constraint prevents it?

In [21]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS products_new(
product_id INTEGER PRIMARY KEY,
product_name TEXT,
category TEXT,
price REAL CHECK(price >= 0)
)
""")

conn.commit()

In [24]:
cursor.execute("""
INSERT INTO products_new
(product_id, product_name, category, price)
VALUES
(10,'Keyboard','Electronics',-50);
""")

conn.commit()

IntegrityError: CHECK constraint failed: price >= 0

# Section B — Filtering & Optimization (WHERE, Indexes)

## Q7. Retrieve all orders with status = 'Delivered'.

### Add status column:

In [25]:
id="bq71"
cursor.execute("""
ALTER TABLE orders
ADD COLUMN status TEXT
""")

conn.commit()

### Update sample values:

In [26]:
id="bq72"
cursor.execute("""
UPDATE orders
SET status='Delivered'
WHERE order_id IN (1,3)
""")

cursor.execute("""
UPDATE orders
SET status='Cancelled'
WHERE order_id=2
""")

conn.commit()

### Query:

In [27]:
id="bq73"
pd.read_sql_query(
"""
SELECT *
FROM orders
WHERE status='Delivered';
""",
conn
)

,order_id,customer_id,order_date,status
0,1,1,2026-01-10,Delivered
1,3,1,2026-02-01,Delivered


# Q8. Find all products in the 'Electronics' category with a unit_price greater than ₹2000.

In [28]:
id="bq81"
pd.read_sql_query(
"""
SELECT *
FROM products
WHERE category='Electronics'
AND price > 2000;
""",
conn
)

,product_id,product_name,category,price
0,1,Laptop,Electronics,60000.0
1,2,Phone,Electronics,30000.0


# Q9. List all customers who joined in the year 2024 and belong to Maharashtra.

In [29]:
id="bq91"
cursor.execute("""
ALTER TABLE customers
ADD COLUMN join_date TEXT
""")

cursor.execute("""
ALTER TABLE customers
ADD COLUMN state TEXT
""")

conn.commit()

### Update data:

In [30]:
id="bq92"
cursor.execute("""
UPDATE customers
SET join_date='2024-05-10',
state='Maharashtra'
WHERE customer_id IN (1,2)
""")

conn.commit()

## Query:

In [31]:
id="bq93"
pd.read_sql_query(
"""
SELECT *
FROM customers
WHERE join_date LIKE '2024%'
AND state='Maharashtra';
""",
conn
)

,customer_id,customer_name,email,city,join_date,state
0,1,Amit,amit@gmail.com,Pune,2024-05-10,Maharashtra
1,2,Sneha,sneha@gmail.com,Mumbai,2024-05-10,Maharashtra


# Q10. Find all orders placed between '2024-08-10' and '2024-08-25' (inclusive) that are NOT cancelled.

In [32]:
id="bq101"
pd.read_sql_query(
"""
SELECT *
FROM orders
WHERE order_date 
BETWEEN '2024-08-10' 
AND '2024-08-25'

AND status != 'Cancelled';
""",
conn
)

,order_id,customer_id,order_date,status


# Q11. Explain what the index idx_orders_date does. How would it improve query performance?

In [33]:
id="bq111"
cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_orders_date
ON orders(order_date);
""")

conn.commit()

### Check index:

In [34]:
id="bq112"
pd.read_sql_query(
"""
PRAGMA index_list(orders);
""",
conn
)

,seq,name,unique,origin,partial
0,0,idx_orders_date,0,c,0
1,1,idx_order_customer,0,c,0


# Q12. Will this query use index?

#### SELECT *
 #### FROM customers
 #### WHERE YEAR(join_date)=2024;

##### No, the index usually will not be used.

##### Reason:

YEAR(join_date) applies a function on the column.

## Index-friendly (SARGable) query:

In [38]:
id="bq121"
pd.read_sql_query(
"""
SELECT *
FROM customers
WHERE join_date >= '2024-01-01'
AND join_date < '2025-01-01';
""",
conn
)

,customer_id,customer_name,email,city,join_date,state
0,1,Amit,amit@gmail.com,Pune,2024-05-10,Maharashtra
1,2,Sneha,sneha@gmail.com,Mumbai,2024-05-10,Maharashtra


# Section C — Aggregation (GROUP BY, SUM, COUNT, AVG, MIN, MAX)

# Q13. Count the total number of orders in the orders table.

In [40]:
pd.read_sql_query(
"""
SELECT 
COUNT(*) AS total_orders
FROM orders;
""",
conn
)

,total_orders
0,3


# Q14. Find the total revenue (SUM of total_amount) from all 'Delivered' orders.

### Add total_amount column:

In [41]:
cursor.execute("""
ALTER TABLE orders
ADD COLUMN total_amount REAL
""")

conn.commit()

### Update sample values:

In [42]:
cursor.execute("""
UPDATE orders
SET total_amount = 70000
WHERE order_id=1
""")

cursor.execute("""
UPDATE orders
SET total_amount = 20000
WHERE order_id=2
""")

cursor.execute("""
UPDATE orders
SET total_amount = 30000
WHERE order_id=3
""")

conn.commit()

### Query:

In [44]:
pd.read_sql_query(
"""
SELECT 
SUM(total_amount) AS total_revenue
FROM orders
WHERE status='Delivered';
""",
conn
)

,total_revenue
0,100000.0


# Q15. Calculate the average unit_price of products in each category.

In [45]:
pd.read_sql_query(
"""
SELECT
category,
AVG(price) AS average_price

FROM products

GROUP BY category;
""",
conn
)

,category,average_price
0,Electronics,45000.0
1,Furniture,7500.0


# Q16. For each order status, find the count of orders and total revenue. Sort by total revenue descending.

In [46]:
pd.read_sql_query(
"""
SELECT

status,

COUNT(order_id) AS total_orders,

SUM(total_amount) AS total_revenue


FROM orders

GROUP BY status

ORDER BY total_revenue DESC;

""",
conn
)

,status,total_orders,total_revenue
0,Delivered,2,100000.0
1,Cancelled,1,20000.0


# Q17. Find the most expensive (MAX) and cheapest (MIN) product in each category.

In [48]:
pd.read_sql_query(
"""
SELECT

category,

MAX(price) AS highest_price,

MIN(price) AS lowest_price


FROM products

GROUP BY category;

""",
conn
)

,category,highest_price,lowest_price
0,Electronics,60000.0,30000.0
1,Furniture,10000.0,5000.0


# Q18. List all product categories where average unit_price is greater than ₹2000.

In [51]:
pd.read_sql_query(
"""
SELECT

category,

AVG(price) AS average_price


FROM products


GROUP BY category


HAVING AVG(price) > 2000;

""",
conn
)

,category,average_price
0,Electronics,45000.0
1,Furniture,7500.0


# Section D — Joins & Relationships

# Q19. Write an INNER JOIN query to display each order along with customer details.

## Show:

### order_id
### order_date
### first_name
### last_name
### total_amount

In [53]:
pd.read_sql_query(
"""
SELECT

orders.order_id,
orders.order_date,
customers.customer_name,
orders.total_amount


FROM orders


INNER JOIN customers

ON orders.customer_id = customers.customer_id;

""",
conn
)

,order_id,order_date,customer_name,total_amount
0,1,2026-01-10,Amit,70000.0
1,2,2026-01-15,Sneha,20000.0
2,3,2026-02-01,Amit,30000.0


# Q20. Using LEFT JOIN, list ALL customers and their orders.

In [54]:
pd.read_sql_query(
"""
SELECT

customers.customer_name,
orders.order_id,
orders.order_date,
orders.total_amount


FROM customers


LEFT JOIN orders

ON customers.customer_id = orders.customer_id;

""",
conn
)

,customer_name,order_id,order_date,total_amount
0,Amit,1.0,2026-01-10,70000.0
1,Amit,3.0,2026-02-01,30000.0
2,Sneha,2.0,2026-01-15,20000.0
3,Rahul,NaN,None,NaN
4,Priya,NaN,None,NaN


# Q21. JOIN orders → order_items → products to show order details.

### First add discount column:

In [55]:
cursor.execute("""
ALTER TABLE order_items
ADD COLUMN discount_pct REAL DEFAULT 0
""")

conn.commit()

### Now query:

In [56]:
pd.read_sql_query(
"""
SELECT

orders.order_id,

products.product_name,

order_items.quantity,

products.price AS unit_price,

order_items.discount_pct


FROM orders


JOIN order_items

ON orders.order_id = order_items.order_id


JOIN products

ON order_items.product_id = products.product_id;

""",
conn
)

,order_id,product_name,quantity,unit_price,discount_pct
0,1,Laptop,2,60000.0,0.0
1,1,Phone,1,30000.0,0.0
2,2,Chair,4,5000.0,0.0
3,3,Table,2,10000.0,0.0


# Q22. Difference between LEFT JOIN and RIGHT JOIN
## LEFT JOIN

#### Returns:

#### all records from left table
#### matching records from right table

### Example:

In [60]:
pd.read_sql_query(
"""
SELECT *

FROM customers

LEFT JOIN orders

ON customers.customer_id = orders.customer_id;

""",
conn
)

,customer_id,customer_name,email,city,join_date,state,order_id,customer_id,order_date,status,total_amount
0,1,Amit,amit@gmail.com,Pune,2024-05-10,Maharashtra,1.0,1.0,2026-01-10,Delivered,70000.0
1,1,Amit,amit@gmail.com,Pune,2024-05-10,Maharashtra,3.0,1.0,2026-02-01,Delivered,30000.0
2,2,Sneha,sneha@gmail.com,Mumbai,2024-05-10,Maharashtra,2.0,2.0,2026-01-15,Cancelled,20000.0
3,3,Rahul,rahul@gmail.com,Delhi,None,None,NaN,NaN,None,None,NaN
4,4,Priya,priya@gmail.com,Pune,None,None,NaN,NaN,None,None,NaN


## RIGHT JOIN

#### Returns:

#### all records from right table
#### matching records from left table

## Example:

In [61]:
# SQLite does not directly support RIGHT JOIN.
pd.read_sql_query(
"""
SELECT *

FROM customers

LEFT JOIN orders

ON customers.customer_id = orders.customer_id;

""",
conn
)



,customer_id,customer_name,email,city,join_date,state,order_id,customer_id,order_date,status,total_amount
0,1,Amit,amit@gmail.com,Pune,2024-05-10,Maharashtra,1.0,1.0,2026-01-10,Delivered,70000.0
1,1,Amit,amit@gmail.com,Pune,2024-05-10,Maharashtra,3.0,1.0,2026-02-01,Delivered,30000.0
2,2,Sneha,sneha@gmail.com,Mumbai,2024-05-10,Maharashtra,2.0,2.0,2026-01-15,Cancelled,20000.0
3,3,Rahul,rahul@gmail.com,Delhi,None,None,NaN,NaN,None,None,NaN
4,4,Priya,priya@gmail.com,Pune,None,None,NaN,NaN,None,None,NaN


## FULL OUTER JOIN
#### Returns:

#### all records from both tables
#### matching rows combined
#### unmatched rows as NULL

Used when you need complete comparison between two tables.

# Q23. Identify all Foreign Key relationships.

In [64]:
pd.read_sql_query(
"""
PRAGMA foreign_key_list(orders);
""",
conn
)

,id,seq,table,from,to,on_update,on_delete,match
0,0,0,customers,customer_id,customer_id,NO ACTION,NO ACTION,NONE


In [63]:
pd.read_sql_query(
"""
PRAGMA foreign_key_list(order_items);
""",
conn
)

,id,seq,table,from,to,on_update,on_delete,match
0,0,0,products,product_id,product_id,NO ACTION,NO ACTION,NONE
1,1,0,orders,order_id,order_id,NO ACTION,NO ACTION,NONE


# Test invalid customer_id

In [69]:
cursor.execute("""
PRAGMA foreign_keys = ON;
""")

In [71]:
cursor.execute("""
INSERT INTO orders
(order_id, customer_id, order_date, status, total_amount)
VALUES
(10,999,'2024-08-20','Delivered',5000);
""")

conn.commit()

IntegrityError: UNIQUE constraint failed: orders.order_id

# Section E — Advanced Concepts (CASE, ACID, Transactions)

 ## Q24. Use CASE to classify products into price tiers.

Rules:

Budget → price < 1000
Mid-Range → price between 1000 and 3000
Premium → price > 3000

In [72]:
pd.read_sql_query(
"""
SELECT

product_name,

price AS unit_price,

CASE

WHEN price < 1000 
THEN 'Budget'


WHEN price BETWEEN 1000 AND 3000
THEN 'Mid-Range'


WHEN price > 3000
THEN 'Premium'


END AS price_tier


FROM products;

""",
conn
)

,product_name,unit_price,price_tier
0,Laptop,60000.0,Premium
1,Phone,30000.0,Premium
2,Chair,5000.0,Premium
3,Table,10000.0,Premium


# Q25. Use CASE inside aggregate function to count Delivered vs Not Delivered orders.

In [73]:
pd.read_sql_query(
"""
SELECT

COUNT(
CASE 
WHEN status='Delivered'
THEN 1
END
) AS Delivered_Orders,


COUNT(
CASE 
WHEN status!='Delivered'
THEN 1
END
) AS Not_Delivered_Orders


FROM orders;

""",
conn
)

,Delivered_Orders,Not_Delivered_Orders
0,4,1


# Q26. Explain ACID properties

# Q26. Explain ACID properties

ACID properties are important rules that make database transactions reliable and safe.

## A - Atomicity

Atomicity means a transaction should complete fully or not happen at all.

Example:
In a bank transfer, money is deducted from one account and added to another account. If adding money fails, the deduction should also be cancelled.

## C - Consistency

Consistency means the database should always remain in a valid state before and after a transaction.

Example:
During a bank transfer, the total balance should remain correct. Money should not be lost or created.

## I - Isolation

Isolation means multiple transactions running at the same time should not affect each other.

Example:
If two users try to purchase the last available product, the database should allow only one successful purchase.

## D - Durability

Durability means once a transaction is committed, the changes are permanently saved.

Example:
After a successful bank transaction, the record remains saved even if the system crashes.


# Q27. Write SQL Transaction (BEGIN → COMMIT / ROLLBACK)

In [78]:
cursor.execute("""
INSERT INTO customers
(customer_id, customer_name, email, city, join_date, state)

VALUES

(102,'Test Customer','test102@gmail.com','Pune','2024-08-20','Maharashtra');

""")

conn.commit()

In [77]:
try:

    cursor.execute("BEGIN TRANSACTION")


    # 1. Insert new order

    cursor.execute("""
    INSERT INTO orders
    (order_id, customer_id, order_date, status, total_amount)

    VALUES

    (1011,102,DATE('now'),'Pending',1598.00)
    """)



    # 2. Insert order items

    cursor.execute("""
    INSERT INTO order_items
    (item_id, order_id, product_id, quantity, discount_pct)

    VALUES

    (101,1011,1,1,0)
    """)



    cursor.execute("""
    INSERT INTO order_items
    (item_id, order_id, product_id, quantity, discount_pct)

    VALUES

    (102,1011,2,1,0)
    """)



    # 3. Update product stock

    # Add stock_qty column if required

    cursor.execute("""
    UPDATE products

    SET stock_qty = stock_qty - 1

    WHERE product_id = 1
    """)



    cursor.execute("""
    UPDATE products

    SET stock_qty = stock_qty - 1

    WHERE product_id = 2
    """)



    # 4. Commit transaction

    conn.commit()

    print("Transaction Completed Successfully")


except Exception as e:

    conn.rollback()

    print("Transaction Failed")

    print(e)

Transaction Failed
FOREIGN KEY constraint failed
